# Elasticidad precio-demanda

En este notebook se calcula la elasticidad propia del producto seleccionado, Pepsi. Este
paso es necesario para entender de qué forma se comporta la demanda de Pepsi ante un cambio
en el precio. El cálculo se realiza de forma progresiva, añadiendo confusores en cada nueva
regresión, para observar cómo se ajusta la elasticidad a medida que el modelo se limpia.

### ¿Qué es la elasticidad precio-demanda?

La elasticidad precio-demanda mide **en qué porcentaje varía la cantidad demandada cuando el
precio cambia un 1%**:

$$\varepsilon = \frac{\%\ \Delta\ \text{cantidad}}{\%\ \Delta\ \text{precio}}$$

Para bienes normales es negativa (si el precio sube, la demanda baja). Si su valor absoluto
es mayor que 1, la demanda es **elástica** (reacciona con fuerza al precio); si es menor que
1, **inelástica**. Se estima con una regresión log-log —ln(cantidad) sobre ln(precio)—, en la
que el coeficiente del precio es directamente la elasticidad, sin transformaciones
adicionales.

### Extensión: elasticidad cruzada

Como extensión se calcula la elasticidad cruzada con el rival directo, Coca-Cola: cómo
responde la demanda de Pepsi a cambios en el precio de Coca. No se trata de hacer más fiable
la elasticidad propia, sino de responder a una pregunta distinta —la relación competitiva
entre ambas marcas— que además arroja luz sobre la propia. El **signo** de la elasticidad
cruzada indica el tipo de relación:

- **Positiva → sustitutos:** si Coca sube de precio, la demanda de Pepsi aumenta (el cliente
  se cambia de marca). Es lo esperable entre dos refrescos rivales.
- **Negativa → complementarios:** ambos productos se consumen juntos, de modo que si uno se
  encarece, cae la demanda del otro (por ejemplo, impresora y tinta).

In [27]:
import pandas as pd
import numpy as np


import statsmodels.formula.api as smf

### Carga de datos

In [28]:
pepsi = pd.read_parquet('../data/processed/pepsi_2l.parquet')
pepsi = pepsi[pepsi['start'] > '1991-01-01']
print(pepsi.shape)
print(pepsi.columns.tolist())

(26663, 15)
['STORE', 'UPC', 'WEEK', 'MOVE', 'QTY', 'PRICE', 'SALE', 'PROFIT', 'DESCRIP', 'SIZE', 'COM_CODE', 'start', 'end', 'special', 'UNIT_PRICE']


In [29]:
pepsi['UNIT_PRICE'] = pepsi['PRICE'] / pepsi['QTY'] # calculamos precio unitario
# aplicamos logaritmo a unidades vendidas y precio unitario para calcular elasticidad
pepsi['ln_q'] = np.log(pepsi['MOVE'])
pepsi['ln_p'] = np.log(pepsi['UNIT_PRICE'])

In [30]:
# Transformamos en variable dummy las promociones y festivos
pepsi['PROMO'] = pepsi['SALE'].notna().astype(int)
pepsi['FESTIVO'] = pepsi['special'].str.strip().notna().astype(int)

# Transformamos variable tiendas en categórica
pepsi['STORE'] = pepsi['STORE'].astype('category')


Para trabajar con las variables `SALE` y `special`, las transformamos en variables binarias, ya que no nos interesa en este momento que es cada cosa, simplemente aislar su efecto de la forma más sencilla posible. En `SALE` se registra el tipo de descuento aplicado al producto y en `special` el tipo de festividad que hay en esa semana.

Finalmente, como las tiendas se registran como número, las transformamos en variable categórica para que llegado el momento de aplicar la regresión, no interprete que una tienda tiene más peso que otra por tener diferente número.

Con el dataset estructurado, pasamos a la fase de regresión, en donde se van a generar varias regresiones añadiendo cada vez más complejidad con el efecto que provocan las promociones y días festivos. De esta forma se puede ir evaluando como cambia la elasticidad.

## Regresión

In [31]:
# inicializamos lista para guardar resultados
resultados = []

# Regresión 1 - Ingenua. Relación precio-cantidad sin ningún control.

m0 = smf.ols('ln_q ~ ln_p', data=pepsi).fit() # instanciamos y ajustamos modelo
resultados.append(('1. Ingenua', m0.params['ln_p']))
print(f"Elasticidad ingenua: {m0.params['ln_p']:.4f}")

Elasticidad ingenua: -4.0700


In [32]:
# Regresión con promoción incluida
m1 = smf.ols('ln_q ~ ln_p + PROMO', data=pepsi).fit()
resultados.append(('2. Promoción', m1.params['ln_p']))
print(f"Elasticidad (con promo): {m1.params['ln_p']:.4f}")
print(f"Efecto promoción: {m1.params['PROMO']:.4f} (p = {m1.pvalues['PROMO']:.3f})")

Elasticidad (con promo): -4.0333
Efecto promoción: 0.0244 (p = 0.017)


Los resultados de esta segunda regresión no son muy prometedores, se esperaba una mayor bajada de la elasticidad pero el efecto ha sido mínimo. Por otro lado, el efecto de la promoción apenas parece tener impacto, lo que contradice en cierta medida los hallazgos del EDA, donde se veía como las semanas con promoción tenían un gran impacto en la ventas.

In [33]:
# Cuánto se solapan precio y promoción?
pepsi.groupby("PROMO")["ln_p"].describe()[["mean", "min", "max"]]

,mean,min,max
PROMO,,,
0,0.399886,-0.235722,0.636577
1,0.162020,-0.385662,0.636577


Como ya hemos codificado el efecto de promoción en binario, vamos a examinar esta variable más de cerca, ya que la variable original en la documentación de los propios datos nos indica que puede quedar algún error en las promociones. Si calculamos el precio medio, se puede observar como aquellos precios con promocion (`1`) tienen un precio mucho más bajo que los precios regulares. Observando el rango de precio, que ambas categorías coincidan exactamente en el valor máximo, nos indica lo que ya se sospechaba e indicaba la documentación, posibles errores en el registro de promociones.

In [34]:
pepsi.groupby("PROMO")["MOVE"].mean()

PROMO
0    155.337092
1    450.739685
Name: MOVE, dtype: float64

In [35]:
# Semanas marcadas como promo pero con precio alto: ¿tienen sentido?
pepsi[pepsi["PROMO"] == 1].sort_values("ln_p", ascending=False)[
    ["STORE", "start", "UNIT_PRICE", "MOVE", "SALE", "PROMO"]
].head(10)

,STORE,start,UNIT_PRICE,MOVE,SALE,PROMO
214820,71,1991-07-11,1.89,62,S,1
213876,62,1991-07-11,1.89,30,S,1
211179,33,1991-07-11,1.89,77,S,1
225045,137,1991-07-11,1.89,48,S,1
218323,93,1991-07-11,1.89,51,S,1
213094,53,1991-07-11,1.89,23,S,1
224222,130,1991-07-11,1.89,157,S,1
210219,14,1991-07-11,1.89,44,S,1
218704,95,1991-07-11,1.89,107,S,1
214435,68,1991-07-11,1.89,87,S,1


Inspeccionando las semanas marcadas como promción podemos ver la causa, hay registros marcados como promción pero que cuentan con el precio más elevado y el número de ventas es bajo.

Para tratar de corregir esta situación vamos a reconstruir la señal de promoción a partir del propio precio. Para reconstruir el precio lo que se hace es definir un **precio de referencia** por tienda, el precio regular del producto en cada tienda, y se mide la **profundiad de descuento** como la caída del precio observado respecto a la referencia. 

Para construir el precio de referencia vamos a emplear un cuantil alto (percentil 90), es que donde se encuentran los precios altos, para una ventana móvil de 13 semanas, haciendolo coincidir con el trimestre, calculado por tienda. La lógica de esta decisión es que como las promociones bajan el precio y son frecuentes en la serie, el precio regular siempre vive en la parte alta de la distribución local. Por tanto un cuantil alto captura mejor que la mediana, que se vería contaminada con la cantidad de semanas que tiene el precio descontado. La ventana móvil permite que la referencia de precio se adapte al cambio que se vaya ocasionando a lo largo de la serie, de esta forma evitamos establecer un precio fijo para los 6 años.

## Feature Engineering

In [36]:
# Nos aseguramos de que todas las tiendas estén bien agrupadas por fecha.
pepsi_sorted = pepsi.sort_values(['STORE','start']).reset_index(drop=True)

# Parámetros de ventana móvil
VENTANA = 13 # semanas coincidente con trimestre
CUANTIL = 0.90 # parte más alta de la distribución

# creamos nueva variable con precio referencia
pepsi_sorted['PRECIO_REF'] = (
    pepsi_sorted.groupby('STORE', observed=True)['UNIT_PRICE'] # aplicamos el cálculo separado por tienda
    .transform(lambda x: x.rolling(VENTANA, min_periods=5, center=True) # Con center indicamos que mire tanto atrás como adelante
               .quantile(CUANTIL))
)

# Comprobamos que los precios se han construido de forma correcta y se encuentran en la parte alta de la distrbución
display(pepsi_sorted['PRECIO_REF'].describe())
display(pepsi_sorted[['UNIT_PRICE', 'PRECIO_REF']].sample(10))
print(f'Precios de referencia vacíos: {pepsi_sorted["PRECIO_REF"].isna().sum()}')


count    26663.000000
mean         1.579495
std          0.108661
min          1.180000
25%          1.490000
50%          1.588000
75%          1.688000
max          1.890000
Name: PRECIO_REF, dtype: float64

,UNIT_PRICE,PRECIO_REF
16264,0.89,1.790
12442,0.99,1.588
11272,1.39,1.390
23853,1.48,1.490
13633,1.33,1.590
22037,0.99,1.588
18866,1.33,1.690
2763,1.49,1.490
803,1.63,1.688
2721,1.09,1.490


Precios de referencia vacíos: 0


Una vez se ha construido el **precio de referencia** y observando los resultados se determina que se ha reconstruido la variable de forma correcta. Los cuartiles de la variable se encuentran en la parte alta de la distrbución del precio original. Si observamos una pequeña muestra, se puede ver como todos los precios de referencia son mayores o iguales al precio original, justo lo que se buscaba. Finalmente hacemos una comprobación rápida para asegurarnos de que no quedaron espacios en blanco dentro del prefio de referencia.

El siguiente paso es calcular el descuento aplicado, es decir, la diferencia entre el precio de referencia y el precio en logaritmo. Como la diferencia de logaritmos es aproximadamente un cambio porcentual, la variable que se va a crear se puede leer directamente como **qué porcentaje por debajo de su precio regular está el precio esta semana**.

Con esta nueva variable será posible arreglar el problema que presentaba la segunda regresión, en donde no se diferenciaba precio de promoción, dando al modelo prácticamente la misma información.

In [37]:
# creamos descuento
pepsi_sorted['DESCUENTO'] = np.log(pepsi_sorted['PRECIO_REF']) - pepsi_sorted['ln_p']

# Comprobamos resutlados
display(pepsi_sorted['DESCUENTO'].describe())
print(f"Cantidad de descuentos negativos: {(pepsi_sorted['DESCUENTO'] < -0.01).sum()}")

count    26663.000000
mean         0.165372
std          0.198597
min         -0.131336
25%          0.000000
50%          0.064958
75%          0.337072
max          0.898486
Name: DESCUENTO, dtype: float64

Cantidad de descuentos negativos: 133


Las estadísticas del descuento nos indican que hay precios negativos, con un mínimo del 13% (-0.13) por encima de su propio precio regular. Si observamos el número total de descuentos negativos obtenemos un resultado de 133, algo que conceptualmente no debería ocurrir. 

Lo más probable es que se hubiese producido un cambio de precio y el precio de referencia creado aún no hubiese capturado el cambio. Antes de seguir vamos a comprobar que el problema está en el cambio de escalón.

In [38]:
pepsi_sorted[pepsi_sorted["DESCUENTO"] < -0.01][["STORE", "start", "UNIT_PRICE", "PRECIO_REF", "DESCUENTO"]].head(10)

,STORE,start,UNIT_PRICE,PRECIO_REF,DESCUENTO
125,2,1993-06-17,1.69,1.530,-0.099461
280,2,1996-08-08,1.69,1.670,-0.011905
640,8,1991-11-07,1.49,1.390,-0.069472
721,8,1993-06-10,1.69,1.650,-0.023953
722,8,1993-06-17,1.69,1.650,-0.023953
877,8,1996-08-08,1.69,1.548,-0.087765
884,8,1996-09-26,1.59,1.536,-0.034552
1040,9,1993-06-10,1.68,1.642,-0.022879
1360,12,1993-06-17,1.59,1.506,-0.054277
1415,12,1994-07-14,1.59,1.570,-0.012658


Comprobando los descuentos que salen en negativo se puede ver como el error se produce por un cambio de precio en `UNITE PRICE` que el precio de referencia no llega a capturar a tiempo, nada grave de lo que preocuparse. Para evitar que provoquen errores en la regresión vamos a llevar estos valores negativos a 0, indicando así que para esas referencias no hubo descuento en esa semana.

In [39]:
pepsi_sorted['DESCUENTO'] = pepsi_sorted['DESCUENTO'].clip(lower=0) # llevamos descuentos negativos a 0
print(f"Cantidad de descuentos negativos: {(pepsi_sorted['DESCUENTO'] < -0.01).sum()}")

Cantidad de descuentos negativos: 0


Ahora que el se ha reconstruido tanto el precio como el descuento ya es posible volver a trabajar en la regresión para calcular la elasticidad y medir el efecto del descuento.

## Regresión con feature engineering

In [40]:
m1b = smf.ols('ln_q ~ ln_p + DESCUENTO', data=pepsi_sorted).fit()
resultados.append(('2. Descuento', m1b.params['ln_p']))
print(f"Elasticidad:        {m1b.params['ln_p']:.4f}")
print(f"Efecto descuento:   {m1b.params['DESCUENTO']:.4f} (p = {m1b.pvalues['DESCUENTO']:.3f})")
print(m1b.summary())

Elasticidad:        -4.1763
Efecto descuento:   -0.1129 (p = 0.064)
                            OLS Regression Results                            
Dep. Variable:                   ln_q   R-squared:                       0.593
Model:                            OLS   Adj. R-squared:                  0.593
Method:                 Least Squares   F-statistic:                 1.943e+04
Date:               sá., 12 sep. 2026   Prob (F-statistic):               0.00
Time:                        11:03:06   Log-Likelihood:                -27074.
No. Observations:               26663   AIC:                         5.415e+04
Df Residuals:                   26660   BIC:                         5.418e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------

Al realizar la regresión con el descuento sobre el precio original, volvemos a tener el mismo problema que en el primer planteamiento, tanto el precio real como el descuento se solapan, dando lugar a que ambas variables comparten la misma información. Si observamos los coeficientes, el DESCUENTO sale con signo negativo. Más que interpretarlo (descontar no puede reducir las ventas, sabemos por el EDA que las dispara), lo leemos como la señal de que esta especificación está mal planteada: al solaparse precio y descuento, el coeficiente del descuento no tiene una interpretación fiable. Tampoco resulta ser un coeficiente significativo.

Antes de sacar conclusiones precipitadas vamos a plantear una fórmula para el modelo diferente, en donde aplicaremos el logaritmo del `PRECIO_REF`, que es la elasticidad respecto al precio regular, es decir, cómo responder la demanda cuando cambia el precio de lista, no el de oferta.

In [41]:
m1c = smf.ols('ln_q ~ np.log(PRECIO_REF) + DESCUENTO', data=pepsi_sorted).fit()
resultados.append(('2. PRECIO_REF', m1c.params['np.log(PRECIO_REF)']))
print(m1c.summary())
print(f"Elasticidad: {m1c.params['np.log(PRECIO_REF)']:.4f}")
print(f"Efecto descuento: {m1c.params['DESCUENTO']:.4f} (p = {m1c.pvalues['DESCUENTO']:.3f})")

                            OLS Regression Results                            
Dep. Variable:                   ln_q   R-squared:                       0.593
Model:                            OLS   Adj. R-squared:                  0.593
Method:                 Least Squares   F-statistic:                 1.943e+04
Date:               sá., 12 sep. 2026   Prob (F-statistic):               0.00
Time:                        11:03:06   Log-Likelihood:                -27076.
No. Observations:               26663   AIC:                         5.416e+04
Df Residuals:                   26660   BIC:                         5.418e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              6.2761      0

Cambiando el modelo, se puede ver como el `DESCUENTO` finalmente cobra sentido, obteniendo un resultado muy significativo en consonancia con los hallazgos obtenidos en el EDA. Ahora se aprecia como un precio con descuento dispara las ventas del producto.

En cuanto al $R^2$ vemos que este sigue siendo idéntico. La mejora no está en que el modelo sea capaz de explicar más la varianza, sino en repartir de forma correcta los efectos entre precio y descuento. De todas formas, este no es un modelo definitivo ya que aún quedan variables pendientes de añadir.

A continuación se añade la variable `FESTIVO` como predictor.

In [42]:
m2 = smf.ols('ln_q ~ np.log(PRECIO_REF) + DESCUENTO + FESTIVO', data=pepsi_sorted).fit()
resultados.append(('3. FESTIVO', m2.params['np.log(PRECIO_REF)']))
print(m2.summary())
print(f"Elasticidad (precio regular): {m2.params['np.log(PRECIO_REF)']:.4f}")
print(f"Efecto descuento: {m2.params['DESCUENTO']:.4f}")
print(f"Efecto festivo: {m2.params['FESTIVO']:.4f} (p = {m2.pvalues['FESTIVO']:.3f})")

                            OLS Regression Results                            
Dep. Variable:                   ln_q   R-squared:                       0.593
Model:                            OLS   Adj. R-squared:                  0.593
Method:                 Least Squares   F-statistic:                 1.297e+04
Date:               sá., 12 sep. 2026   Prob (F-statistic):               0.00
Time:                        11:03:06   Log-Likelihood:                -27067.
No. Observations:               26663   AIC:                         5.414e+04
Df Residuals:                   26659   BIC:                         5.418e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              6.2843      0

El efecto festivo, a pesar de ser significativo, tiene un impacto muy poco relevante, se puede interpretar como que en una semana con festivo hay un -3% menos de ventas. En este caso, como el efecto no importa mucho, es una variable candidata a ser eliminada del modelo final, de esta forma se simplifica el modelo. 

Antes de eliminar variables, queda pendiente diferenciar por tienda, que es el siguiente paso.

In [43]:
m3 = smf.ols('ln_q ~ np.log(PRECIO_REF) + DESCUENTO + FESTIVO + STORE', data=pepsi_sorted).fit()
resultados.append(('4. STORE', m3.params['np.log(PRECIO_REF)']))
print(m3.summary())
print(f"Elasticidad (precio regular): {m3.params['np.log(PRECIO_REF)']:.4f}")
print(f"Efecto descuento: {m3.params['DESCUENTO']:.4f}")
print(f"Efecto festivo: {m3.params['FESTIVO']:.4f}")

                            OLS Regression Results                            
Dep. Variable:                   ln_q   R-squared:                       0.734
Model:                            OLS   Adj. R-squared:                  0.733
Method:                 Least Squares   F-statistic:                     772.4
Date:               sá., 12 sep. 2026   Prob (F-statistic):               0.00
Time:                        11:03:06   Log-Likelihood:                -21399.
No. Observations:               26663   AIC:                         4.299e+04
Df Residuals:                   26567   BIC:                         4.378e+04
Df Model:                          95                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              5.8692      0

Al añadir la tienda como variable diferenciadora, la elasticidad baja de -4.26 a -3.78, con un aumento de $R^2$. Al mismo tiempo, el efecto del `DESCUENTO` también se reduce ligeramente pero sigue siendo muy fuerte. 

Aún así, la elasticidad obtenida sigue siendo muy elevada. Esto no se debe a un defecto del modelo, sino a que tipo de elasticidad estamos midiendo. Hasta ahora hemos trabajado con una elasticidad de marca, es decir midiendo la respuesta de la demanda al cambio en su propio precio, sin tener en cuenta productos sustitutivos. En la realidad, si el precio de un producto sube mucho, el consumidor procederá a cambiar a otro producto similar con un precio mucho menor. Distinta sería la situación si midiésemos la elasticidad de categoría, que se da al medir todos los refrescos a la vez y por tanto no hay sustituto posible.

Por tanto, el resultado obtenido, mide la pérdida de Pepsi frente a sus rivales ante una subida en solitario. La forma natural de completar el cuadro es incorporar el precio de los sustitutos para estimar la **elasticidad cruzada**, situación que se aborda con la extensión del notebook.

## Extensión elasticidad cruzada

In [44]:
# carga dataset y filtrado para cocacola
cat = pd.read_parquet('../data/interim/soft_drinks_enriched.parquet')

coca = cat[cat['UPC'] == 4900000639].copy()
print(coca.shape)
print(f"Fecha min: {coca['start'].min()}; Fecha max: {coca['start'].max()}")
coca['STORE'].nunique()

(31667, 14)
Fecha min: 1989-09-14 00:00:00; Fecha max: 1997-05-01 00:00:00


93

Con los datos cargados procedemos a reconstruir el precio de cocacola como hicimos con pepsi previamente.

In [45]:
# FIltramos fecha para que tengan misma longitud
coca = coca[coca['start'] > '1991-01-01']

# Ordenamos por tienda y fecha
coca_sorted = coca.sort_values(['STORE', 'start']).reset_index(drop=True)

# Calculamos precio unitario
coca_sorted['UNIT_PRICE'] = coca_sorted['PRICE'] / coca_sorted['QTY']

# Calculamos precio de referencia con cuantil MOVIL
coca_sorted['PRECIO_REF_COCA'] = (
    coca_sorted.groupby('STORE', observed=True)['UNIT_PRICE']
    .transform(lambda x: x.rolling(VENTANA, min_periods=5, center=True).quantile(CUANTIL))
)

coca_precio = coca_sorted[['STORE', 'start', 'PRECIO_REF_COCA']].copy()
print(coca_precio.shape)
display(coca_precio.head())
display(coca_precio['PRECIO_REF_COCA'].describe())

(26577, 3)


,STORE,start,PRECIO_REF_COCA
0,2,1991-01-03,1.89
1,2,1991-01-10,1.89
2,2,1991-01-17,1.89
3,2,1991-01-24,1.89
4,2,1991-01-31,1.89


count    26577.000000
mean         1.575577
std          0.107444
min          1.180000
25%          1.490000
50%          1.581000
75%          1.680000
max          1.890000
Name: PRECIO_REF_COCA, dtype: float64

In [46]:
pepsi_sorted = pepsi_sorted.merge(coca_precio, on=['STORE', 'start'], how='left')

print(f'Filas totales: {len(pepsi_sorted)}')
print(f"Filas sin precio en Coca: {pepsi_sorted['PRECIO_REF_COCA'].isna().sum()}")
print(f"Fracción con dato: {pepsi_sorted['PRECIO_REF_COCA'].notna().mean()}")

Filas totales: 26663
Filas sin precio en Coca: 88
Fracción con dato: 0.9966995461876008


In [47]:
pepsi_sorted['STORE'] = pepsi_sorted['STORE'].astype('category')

m_cross = smf.ols(
    'ln_q ~ np.log(PRECIO_REF) + np.log(PRECIO_REF_COCA) + DESCUENTO + FESTIVO  + STORE', data=pepsi_sorted
).fit()

print(f"Elasticidad propia (Pepsi): {m_cross.params['np.log(PRECIO_REF)']:.4f}")
print(f"Elasticidad cruzada (Coca): {m_cross.params['np.log(PRECIO_REF_COCA)']:.4f}")
print(f"Efecto descuento: {m_cross.params['DESCUENTO']:.4f}")

print(m_cross.summary())

Elasticidad propia (Pepsi): -4.6231
Elasticidad cruzada (Coca): 0.9144
Efecto descuento: 4.0490
                            OLS Regression Results                            
Dep. Variable:                   ln_q   R-squared:                       0.735
Model:                            OLS   Adj. R-squared:                  0.734
Method:                 Least Squares   F-statistic:                     765.6
Date:               sá., 12 sep. 2026   Prob (F-statistic):               0.00
Time:                        11:03:07   Log-Likelihood:                -21247.
No. Observations:               26575   AIC:                         4.269e+04
Df Residuals:                   26478   BIC:                         4.348e+04
Df Model:                          96                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
----------------------

Para cerrar el estudio de la elasticidad, se incorporó el precio rgular de Coca-Cola, el sustituto directo de Pepsi, reconstruyendo los precios de la misma forma (cuantil 90 con ventana móvil de 13 semanas por tienda). El resultado obtenido, confirma la relación de sustitución entre ambas marcas y releva datos importantes sobre la elasticidad propia.

La forma de interpretar los coeficientes sería:
- Elasticidad propia de Pepsi: -4.62. Si el precio regular de Pepsi sube un 1% su demanda cae un 4.62%. Estamos ante una elasticidad elevada, coherente con un mercado con productos sustitutos cercanos.
- Elasticidad cruzada con Coca-Cola: 0.91. Si el precio de Coca-Cola sube un 1% entonces la demanda de Pepsi aumenta un 0.91%. El signo positivo confirma que son sustitutos, al encarecerse la Coca-Cola, parte de los clientes comprarán Pepsi
- Efecto descuento: 4.05. Un descuento del 1% sobre el precio regular de Pepsi eleva la demanda un 4.05%, en línea con los hallazgos encontrados en el EDA.

Al incluir Coca-Cola, la elasticidad de Pepsi no se atenúa, al contrario, se acentúa. El motivo es que los precios de Pepsi y Coca-Cola tienen a moverse juntos. Sin controlar el precio rival, el modelo mezcla dos situaciones distitnas: solo sube Pepsi y suben ambas a la vez, donde el cliente tiene menos incentivo a cambiar de marca. Al aislar el precio de Coca-COla, emerge la respuesta real de la demanda a un cambio de precio de Pepsi en solitario, que es más intensa.

